# 03 - The submission, the one-shot score, and what it means

This notebook is the only place you should read holdout-derived numbers, and it
reads them from the frozen artifact produced by `scripts/final_score.py`, which
ran exactly once.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 120)
ART = ROOT / "reports" / "artifacts"
print("project root:", ROOT)

project root: C:\Z_Coding\Cloud-Workload-Forecasting


## The submission

Three columns, 24 rows, timestamps inferred from the data.

In [2]:
preds = pd.read_csv(ROOT / "predictions.csv")
print(preds.head(3).to_string(index=False))
print("...")
print(preds.tail(3).to_string(index=False))
print("rows:", len(preds), "unique timestamps:", preds.timestamp.nunique())

 timestamp  active_vm_count  aggregate_cpu_utilization
     33000            10000                   9.164556
     33300            10000                   9.164556
     33600            10000                   9.164556
...
 timestamp  active_vm_count  aggregate_cpu_utilization
     39300            10000                   9.164556
     39600            10000                   9.164556
     39900            10000                   9.164556
rows: 24 unique timestamps: 24


Every value is the same because the selected model is persistence on a series
whose training mean is flat. That is not a bug; it is what the validation chose.

## One-shot scores

Reported as-is. The submitted model lost to a moving average on the holdout.

In [3]:
scores = pd.read_csv(ART / "final_scores.csv")
scores.sort_values(["target", "rmse"]).round(4).to_string(index=False)

'                   target            candidate   rmse    mae   bias   mase  scale_naive_mae_train best_baseline  best_baseline_rmse  skill_vs_best_baseline  rmse_improvement_vs_best_pct  rmse_improvement_vs_persistence_pct\n          active_vm_count          persistence 0.0000 0.0000 0.0000    NaN                 0.0000   persistence              0.0000                     NaN                           NaN                                  NaN\n          active_vm_count baseline:persistence 0.0000 0.0000 0.0000    NaN                 0.0000   persistence              0.0000                     NaN                           NaN                                  NaN\n          active_vm_count         baseline:ma6 0.0000 0.0000 0.0000    NaN                 0.0000   persistence              0.0000                     NaN                           NaN                                  NaN\n          active_vm_count        baseline:ma12 0.0000 0.0000 0.0000    NaN                 0.0000   per

Read that table twice. On the training folds persistence beat `ma24`
(0.2123 vs 0.2927). On the 24 holdout steps `ma24` beat persistence
(0.1537 vs 0.1848), a skill score of about -0.20 for the submission.

We did not retune. `docs/protocol.md` says the holdout is scored once and the
model is not changed afterwards, and a result you can only trust when it is
favourable is not a result.

## Why it happened

The holdout window drifted slightly upward relative to the last training day. A
24-step average sits closer to that level than the last single observation does.
That is luck, not skill - `ma24` had no way to know. With five overlapping
origins the standard error was wider than the gap between candidates, so the fold
ranking never had the resolution to separate them.

## Post-hoc analysis (labelled)

Everything below is post-hoc: it explains the result and changes nothing.

In [4]:
print(json.dumps(json.loads((ART / "posthoc_summary.json").read_text()), indent=2)[:1500])

{
  "posthoc": true,
  "target": "aggregate_cpu_utilization",
  "n_holdout": 24,
  "rmse": 0.18478539074333672,
  "mae": 0.14678245915042334,
  "bias": 0.1044315580662661,
  "holdout_mean_actual": 9.268987733051413,
  "holdout_mean_predicted": 9.164556174985147,
  "holdout_min": 8.983613452960672,
  "holdout_max": 9.63599020869164,
  "train_mean": 9.414089019539007,
  "train_mean_last_24": 9.249522346265591,
  "peak_threshold_train_pp": 9.668361517488389,
  "lag24_autocorrelation_train": -0.026381275989692133
}


In [5]:
print("errors by horizon (post-hoc):")
print(pd.read_csv(ART / "posthoc_errors_by_horizon.csv").round(4).to_string(index=False))
print()
print("interval coverage (post-hoc):")
print(pd.read_csv(ART / "posthoc_interval_coverage.csv").round(4).to_string(index=False))

errors by horizon (post-hoc):
 horizon_step  minutes_ahead  actual  predicted  error_actual_minus_pred  abs_error
            1            5.0  9.1685     9.1646                   0.0039     0.0039
            2           10.0  8.9836     9.1646                  -0.1809     0.1809
            3           15.0  9.0432     9.1646                  -0.1213     0.1213
            4           20.0  9.2735     9.1646                   0.1090     0.1090
            5           25.0  9.0775     9.1646                  -0.0870     0.0870
            6           30.0  9.0463     9.1646                  -0.1182     0.1182
            7           35.0  9.1639     9.1646                  -0.0007     0.0007
            8           40.0  9.2471     9.1646                   0.0825     0.0825
            9           45.0  9.1876     9.1646                   0.0230     0.0230
           10           50.0  9.2322     9.1646                   0.0677     0.0677
           11           55.0  9.2543     9.164

The uncertainty statement held up better than the point forecast: the nominal 90%
band covered 91.7% of holdout steps, and every miss was above the band (none
below), consistent with the small upward bias in the forecast.

## The capacity decision

A forecast is not a decision. Provisioning uses a quantile of the band, chosen by
the cost of being short versus long.

In [6]:
print(json.loads((ART / "policy_choice.json").read_text()))
print()
print(pd.read_csv(ART / "policy_holdout_evaluation.csv").round(4).to_string(index=False))

{'target': 'aggregate_cpu_utilization', 'model': 'persistence', 'c_under': 8.0, 'c_over': 1.0, 'newsvendor_level': 0.8888888888888888, 'chosen_level': 0.6, 'interval_method': 'empirical', 'widen_factor': 3.0, 'peak_threshold_train_pp': 9.668361517488389, 'bench_width_median_abs_diff_pp': 0.10317268365704457, 'peak_risk_on_holdout_band': {'max_probability_of_peak': 0.11179867013930589, 'steps_with_peak_probability_above_threshold': 0, 'band_unusually_wide': True, 'action': 'hold extra headroom'}, 'note': 'Level and margin chosen on rolling-origin folds inside the training window; the holdout is scored once by scripts/final_score.py.'}

 level  c_under  c_over  shortfall_events  n_steps  mean_shortfall_units  mean_overprovision_units  max_shortfall_units  net_cost_per_step        policy
  0.50      8.0     1.0                19       24                0.1256                    0.0212               0.4714             1.0260           0.5
  0.60      8.0     1.0                12       24 

In [7]:
sens = pd.read_csv(ART / "policy_holdout_sensitivity.csv")
sens.pivot_table(index="cost_ratio", columns="policy", values="net_cost_per_step").round(3)

policy,0.5,0.6,0.7,0.75,0.8,0.85,0.9,0.95,mean_forecast
cost_ratio,,,,,,,,,
1.0,0.147,0.118,0.134,0.152,0.185,0.234,0.309,0.427,0.147
2.0,0.272,0.189,0.169,0.172,0.193,0.236,0.309,0.427,0.272
4.0,0.524,0.329,0.239,0.211,0.211,0.241,0.309,0.427,0.524
8.0,1.026,0.611,0.378,0.289,0.246,0.249,0.309,0.427,1.026
16.0,2.031,1.174,0.657,0.445,0.316,0.266,0.309,0.427,2.031
32.0,4.041,2.300,1.214,0.757,0.455,0.299,0.309,0.427,4.041


The optimal provisioning level moves from the 50th percentile at a 1:1 cost ratio
to the 85th at 32:1. There is no universal headroom rule; there is only the rule
your cost structure implies. Saying that out loud is more useful than quoting a
single "safe" percentage.

## Operational takeaway

1. The submitted forecast is beatable, and we say so.
2. The uncertainty band was better calibrated than the point forecast and is what
   the capacity decision actually uses.
3. The mean is 9.4 pp while the cross-VM P95 is 37.5 pp, so any plan built on the
   aggregate alone under-provisions the tail.